# G1 Academy 5 - Solution: services, ROS 2, audio, and chatbot


## Introduction
Use the setup scripts for CycloneDDS/Unitree SDK, Wi-Fi, and RealSense ZMQ streaming. This notebook uses native rclpy for a dedicated demo ROS 2 topic, native DDS subscribers for robot audio, and supplied ASR/OpenAI modules for the heavy integrations. Never publish a demo node to lowcmd/Dex3 command topics.

## Deployment and ROS 2 boundaries

The environment, Wi-Fi, and RGB-D scripts are operator-reviewed setup examples, not notebook cells to run blindly. Confirm the Jetson's ROS distribution, Python version, CycloneDDS library path, interface, and dashboard/service health before connecting a native SDK client.

A ROS 2 learning package uses rclpy topics that are separate from the robot's DDS command channels. It is appropriate to publish /academy/demo_status; it is not appropriate to publish rt/lowcmd or Dex3 command topics. Audio observation uses the native Unitree DDS String topic after wake-up mode is enabled externally. The provided ASR/OpenAI integration consumes that payload; participants own safe prompting, result validation, and speech handoff.


## Task 1 - Create a ROS 2 package demo
Imports: rclpy provides node lifecycle; Node creates publishers/subscribers/timers; String is a typed ROS message. Build ament_python package academy_topic_demo, add this node as console script, then use only /academy/demo_status.


In [ ]:
import rclpy
from rclpy.node import Node
from std_msgs.msg import String
class DemoNode(Node):
    def __init__(self):
        super().__init__("academy_demo")
        self.pub=self.create_publisher(String, "/academy/demo_status", 10)
        self.sub=self.create_subscription(String, "/academy/demo_status", self.on_status, 10)
        self.timer=self.create_timer(1.0, self.publish_status)
    def publish_status(self):
        msg=String(); msg.data="academy demo alive"; self.pub.publish(msg)
    def on_status(self, msg): self.get_logger().info(msg.data)
# ros2 pkg create --build-type ament_python academy_topic_demo --dependencies rclpy std_msgs


## Task 2 - Build a minimal direct-SDK Robot class

Create a small Robot class similar in shape to sdk_wrapper, but keep native clients visible. It should own one ChannelFactory configuration, initialize LocoClient, MotionSwitcherClient, and AudioClient once, and expose a conservative state/stop method. This class is the bridge from the notebook helpers to the final participant-built wrapper.


In [ ]:
from unitree_sdk2py.core.channel import ChannelFactoryInitialize
from unitree_sdk2py.g1.loco.g1_loco_client import LocoClient
from unitree_sdk2py.comm.motion_switcher.motion_switcher_client import MotionSwitcherClient
from unitree_sdk2py.g1.audio.g1_audio_client import AudioClient

class Robot:
    _factory_config = None
    def __init__(self, interface="eth0", domain_id=0):
        config=(int(domain_id), str(interface))
        if Robot._factory_config is None:
            ChannelFactoryInitialize(*config); Robot._factory_config=config
        elif Robot._factory_config != config:
            raise RuntimeError("Restart kernel before changing DDS interface/domain.")
        self.loco=LocoClient(); self.loco.SetTimeout(5.0); self.loco.Init()
        self.switcher=MotionSwitcherClient(); self.switcher.SetTimeout(5.0); self.switcher.Init()
        self.audio=AudioClient(); self.audio.SetTimeout(5.0); self.audio.Init()
    def stop(self):
        return self.loco.StopMove()
    def state_summary(self):
        code, mode = self.switcher.CheckMode()
        return {"motion_code": int(code), "motion_mode": mode}

# robot = Robot(); print(robot.state_summary())


## Task 3 - Observe audio, then call supplied ASR/OpenAI integration

First create a native DDS subscriber for rt/audio_msg after wake-up mode is enabled through the app/remote. Cache the newest payload. Then pass it to the provided ASR/OpenAI integration and use the Robot.audio client with util.play_piper_text to speak a short reply.


In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelSubscriber
from unitree_sdk2py.idl.std_msgs.msg.dds_ import String_
from util import load_provided_pipeline, play_piper_text
latest_audio = {"text": None, "timestamp": 0.0}
def on_audio(message):
    latest_audio["text"] = message.data; latest_audio["timestamp"] = time.time()
audio_sub = ChannelSubscriber("rt/audio_msg", String_); audio_sub.Init(on_audio, 20)
def voice_chat_turn(robot, integration_module="academy_asr_openai", language="en"):
    if not latest_audio["text"] or time.time()-latest_audio["timestamp"] > 3.0:
        raise RuntimeError("No fresh wake-up audio message.")
    integration=load_provided_pipeline(integration_module)
    reply=integration.chat(integration.transcribe(latest_audio["text"]))
    play_piper_text(robot.audio, reply, language=language)
    return reply


### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.
